In [13]:
# !pip install openrouteservice
# !pip install tqdm

In [14]:
import pandas as pd
import openrouteservice
from openrouteservice import convert
import time
from tqdm import tqdm

In [15]:
# Example DataFrame
df = pd.DataFrame({
    'event_id': [1, 2, 3],
    'latitude': [48.86137, 48.82084, 48.853],
    'longitude': [2.3433, 2.3682, 2.3499]
})

# Origin
origin = [2.2802465, 48.9073522]  # [lon, lat]

# Destinations
destinations = df[["longitude", "latitude"]].values.tolist()

# ORS client
client = openrouteservice.Client(key='5b3ce3597851110001cf6248386184f778cf4a6d819d5a4422fad8dc')

# Function to get travel times for a given profile
def get_travel_times(profile_name):
    matrix = client.distance_matrix(
        locations=[origin] + destinations,
        profile=profile_name,
        metrics=['duration'],
        sources=[0],
        destinations=list(range(1, len(destinations) + 1))
    )
    durations = matrix['durations'][0]
    return [round(d/60, 1) if d is not None else None for d in durations]

# Driving
df['time_driving_min'] = get_travel_times('driving-car')

# Walking
df['time_walking_min'] = get_travel_times('foot-walking')

# Cycling
df['time_cycling_min'] = get_travel_times('cycling-regular')

print(df)

   event_id  latitude  longitude  time_driving_min  time_walking_min  \
0         1  48.86137     2.3433              28.8              95.2   
1         2  48.82084     2.3682              37.2             161.6   
2         3  48.85300     2.3499              28.3             110.2   

   time_cycling_min  
0              31.2  
1              51.5  
2              35.5  


In [16]:
events = pd.read_csv('events_melted.csv')

C:\Users\clair\AppData\Local\Temp\ipykernel_15656\859874516.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  events = pd.read_csv('events_melted.csv')


In [17]:
events_location = events[['event_id', 'latitude', 'longitude']]

In [18]:
events_location = events_location.drop_duplicates().reset_index(drop=True)

In [19]:
events_location

,event_id,latitude,longitude
0,4283,48.890747,2.340517
1,51498,48.855376,2.362577
2,53204,48.892246,2.387813
3,32382,48.849256,2.412159
4,28184,48.870106,2.323682
...,...,...,...
2135,68152,48.826787,2.366440
2136,68077,48.888952,2.392454
2137,66528,48.880517,2.311239
2138,68227,48.859422,2.307340


In [20]:

# Origin
origin = [2.2802465, 48.9073522]  # [lon, lat]

# Destinations
destinations = events_location[["longitude", "latitude"]].values.tolist()

# ORS client
client = openrouteservice.Client(key='5b3ce3597851110001cf6248386184f778cf4a6d819d5a4422fad8dc')

# Function to get travel times for a given profile
def get_travel_times(profile_name):
    matrix = client.distance_matrix(
        locations=[origin] + destinations,
        profile=profile_name,
        metrics=['duration'],
        sources=[0],
        destinations=list(range(1, len(destinations) + 1))
    )
    durations = matrix['durations'][0]
    return [round(d/60, 1) if d is not None else None for d in durations]

# Driving
events_location['time_driving'] = get_travel_times('driving-car')

# Walking
events_location['time_walking'] = get_travel_times('foot-walking')

# Cycling
events_location['time_cycling'] = get_travel_times('cycling-regular')

print(events_location)

      event_id   latitude  longitude  time_driving  time_walking  time_cycling
0         4283  48.890747   2.340517          20.1          67.5          21.5
1        51498  48.855376   2.362577          30.7         114.8          36.4
2        53204  48.892246   2.387813          26.4         112.6          35.8
3        32382  48.849256   2.412159          32.3         157.9          47.0
4        28184  48.870106   2.323682          20.5          72.6          26.3
...        ...        ...        ...           ...           ...           ...
2135     68152  48.826787   2.366440          39.0         150.4          49.0
2136     68077  48.888952   2.392454          23.6         117.0          37.8
2137     66528  48.880517   2.311239          15.7          55.5          18.1
2138     68227  48.859422   2.307340          24.2          84.6          26.5
2139     68235  48.824389   2.313944          29.9         143.4          49.8

[2140 rows x 6 columns]


In [21]:
events_location

,event_id,latitude,longitude,time_driving,time_walking,time_cycling
0,4283,48.890747,2.340517,20.1,67.5,21.5
1,51498,48.855376,2.362577,30.7,114.8,36.4
2,53204,48.892246,2.387813,26.4,112.6,35.8
3,32382,48.849256,2.412159,32.3,157.9,47.0
4,28184,48.870106,2.323682,20.5,72.6,26.3
...,...,...,...,...,...,...
2135,68152,48.826787,2.366440,39.0,150.4,49.0
2136,68077,48.888952,2.392454,23.6,117.0,37.8
2137,66528,48.880517,2.311239,15.7,55.5,18.1
2138,68227,48.859422,2.307340,24.2,84.6,26.5


In [22]:
events_location.to_csv('travel_times.csv',  index=False)